# Assignment 2: Email Spam Detection using KNN and SVM

**Aim:** Classify emails into Not Spam and Spam using binary classification, then compare K-Nearest Neighbors (KNN) and Support Vector Machine (SVM).

**Dataset:** Download `emails.csv` from the Kaggle link in the assignment and place it beside this notebook. If the dataset has different column names, set them in the configuration cell.

## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             precision_score, recall_score, f1_score)

DATA_PATH = "emails.csv"   # Keep emails.csv in the same folder as this notebook

## 2. Load and inspect dataset
The notebook tries to detect common text and label column names. If automatic detection is incorrect, set `TEXT_COLUMN` and `TARGET_COLUMN` manually.

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
print("Columns:", df.columns.tolist())
display(df.isnull().sum())

In [ ]:
# Automatic column detection (edit these if your dataset uses different names)
text_candidates = ["text", "email", "message", "email_text", "content", "v2"]
target_candidates = ["spam", "label", "target", "class", "category", "v1"]

TEXT_COLUMN = next((c for c in text_candidates if c in df.columns), None)
TARGET_COLUMN = next((c for c in target_candidates if c in df.columns and c != TEXT_COLUMN), None)

print("Detected text column:", TEXT_COLUMN)
print("Detected target column:", TARGET_COLUMN)

if TEXT_COLUMN is None or TARGET_COLUMN is None:
    raise ValueError("Could not detect text/label columns. Set TEXT_COLUMN and TARGET_COLUMN manually.")

## 3. Preprocess data
Missing email text is replaced with an empty string. Labels are converted to binary values: 0 = Not Spam and 1 = Spam. The mapping below supports common labels such as spam/ham and 0/1.

In [ ]:
data = df[[TEXT_COLUMN, TARGET_COLUMN]].copy()
data = data.drop_duplicates()
data[TEXT_COLUMN] = data[TEXT_COLUMN].fillna("").astype(str).str.strip()
data = data[data[TEXT_COLUMN] != ""]

raw_labels = data[TARGET_COLUMN].astype(str).str.strip().str.lower()
label_map = {
    "spam": 1, "1": 1, "yes": 1, "true": 1, "junk": 1,
    "ham": 0, "not spam": 0, "non-spam": 0, "0": 0, "no": 0, "false": 0
}
mapped = raw_labels.map(label_map)

if mapped.isna().any():
    # If labels are already numeric but represented differently, try numeric conversion.
    numeric_labels = pd.to_numeric(data[TARGET_COLUMN], errors="coerce")
    if numeric_labels.notna().all() and set(numeric_labels.unique()).issubset({0, 1}):
        mapped = numeric_labels.astype(int)
    else:
        raise ValueError("Unrecognized target labels. Inspect unique labels and edit label_map.")
data["label"] = mapped.astype(int)

print(data["label"].value_counts().rename(index={0:"Not Spam", 1:"Spam"}))

## 4. Split data and convert email text into numerical features
TF-IDF converts words into numeric features. The vectorizer is fitted only on the training data to avoid data leakage.

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    data[TEXT_COLUMN], data["label"], test_size=0.20,
    random_state=42, stratify=data["label"]
)

tfidf = TfidfVectorizer(stop_words="english", max_features=5000, ngram_range=(1, 2))
X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)

## 5. K-Nearest Neighbors classifier

In [ ]:
# Brute-force search is reliable for sparse TF-IDF features.
knn = KNeighborsClassifier(n_neighbors=5, metric="cosine", algorithm="brute")
knn.fit(X_train, y_train)
knn_pred = knn.predict(X_test)

## 6. Support Vector Machine classifier

In [ ]:
svm = LinearSVC(random_state=42)
svm.fit(X_train, y_train)
svm_pred = svm.predict(X_test)

## 7. Evaluate and compare models
Accuracy, precision, recall and F1-score are reported. The confusion matrix shows correct and incorrect predictions.

In [ ]:
def evaluate_model(name, actual, predicted):
    return {
        "Model": name,
        "Accuracy": accuracy_score(actual, predicted),
        "Precision": precision_score(actual, predicted, zero_division=0),
        "Recall": recall_score(actual, predicted, zero_division=0),
        "F1 Score": f1_score(actual, predicted, zero_division=0)
    }

comparison = pd.DataFrame([
    evaluate_model("K-Nearest Neighbors", y_test, knn_pred),
    evaluate_model("Support Vector Machine", y_test, svm_pred)
]).set_index("Model")
display(comparison)
print("\\nKNN classification report:")
print(classification_report(y_test, knn_pred, target_names=["Not Spam", "Spam"], zero_division=0))
print("\\nSVM classification report:")
print(classification_report(y_test, svm_pred, target_names=["Not Spam", "Spam"], zero_division=0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, pred, title in zip(axes, [knn_pred, svm_pred], ["KNN Confusion Matrix", "SVM Confusion Matrix"]):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Not Spam", "Spam"],
                yticklabels=["Not Spam", "Spam"], ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

## 8. Conclusion
Both KNN and SVM were trained to classify emails as **Not Spam (0)** or **Spam (1)**. Compare their accuracy, precision, recall and F1-score. In spam filtering, recall for the Spam class is important because a false negative means a spam email is missed; precision is also important because a false positive may hide a genuine email. The better model depends on the measured results on this dataset.